In [0]:
# # Updated hr_silver.py
# # Changes: Made it streaming-compatible by splitting into a streaming clean/transform table and using dlt.apply_changes for deduplication.
# # This mimics SCD Type 1 to keep the "best" record per EmpID (highest YearsAtCompany, then highest ingest_timestamp).
# # Moved fillna and standardization before dedup. Added expectations for data quality like in the addresses reference.
# # Dropped unused columns early.

# # Databricks notebook source
# import dlt
# import pyspark.sql.functions as F
# from pyspark.sql.window import Window
# from pyspark.sql.types import NumericType

# # Helper to get numeric columns
# def get_numeric_cols(df):
#     return [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]

# @dlt.table(
#     name="silver_hr_analytics_clean",
#     comment="Cleaned and transformed HR analytics data before dedup (streaming)",
#     table_properties={"quality": "silver"}
# )
# @dlt.expect_or_fail("valid_emp_id", "EmpID IS NOT NULL")
# @dlt.expect_or_drop("valid_department", "Department IS NOT NULL")
# @dlt.expect("valid_monthly_income", "MonthlyIncome > 0")
# def silver_hr_analytics_clean():
#     # Read Bronze (streaming)
#     bronze_df = dlt.read_stream("bronze_hr_analytics")  # Use read_stream for streaming propagation

#     # Replace NULLs with 0 for all numeric columns
#     numeric_cols = get_numeric_cols(bronze_df)
#     if numeric_cols:
#         df = bronze_df.fillna(0, subset=numeric_cols)

#     # Simple standardization
#     df = df.withColumn("Attrition", F.upper(F.col("Attrition")))

#     # Simple derived columns
#     df = (
#         df.withColumn("is_attrited", F.col("Attrition") == "YES")
#           .withColumn("AnnualIncome", F.col("MonthlyIncome") * F.lit(12))
#           .withColumn(
#               "TenureBucket",
#               F.when(F.col("YearsAtCompany") < 3, "New")
#                .when(F.col("YearsAtCompany") < 7, "Experienced")
#                .otherwise("Veteran")
#           )
#     )

#     # Remove obviously unused columns
#     df = df.drop("EmployeeCount", "StandardHours")

#     return df

# # Create the target table for apply_changes (SCD Type 1 equivalent for dedup)
# dlt.create_streaming_table(
#     name="silver_hr_analytics",
#     comment="Deduplicated HR analytics data with latest per EmpID (Silver layer)",
#     table_properties={"quality": "silver"}
# )

# # Apply changes to deduplicate (keep record with max YearsAtCompany, then max ingest_timestamp)
# dlt.apply_changes(
#     target="silver_hr_analytics",
#     source="silver_hr_analytics_clean",
#     keys=["EmpID"],
#     sequence_by=F.struct(F.col("YearsAtCompany"), F.col("ingest_timestamp")),
#     stored_as_scd_type="2"  # Type 1: Overwrite with latest
# )
     

In [0]:
# Databricks notebook source
import dlt
import pyspark.sql.functions as F
from pyspark.sql.types import NumericType, StringType

# -------------------------------
# Helper Functions
# -------------------------------
def get_numeric_cols(df):
    return [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]

def get_string_cols(df):
    return [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

# -------------------------------
# SILVER TABLE: Cleaned HR Data
# -------------------------------
@dlt.table(
    name="silver_hr_analytics",
    comment="Cleaned HR analytics data with NULL handling and derived columns (Silver layer)",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_fail("valid_emp_id", "EmpID IS NOT NULL")
@dlt.expect_or_drop("valid_department", "Department IS NOT NULL")
@dlt.expect("valid_monthly_income", "MonthlyIncome > 0")
def silver_hr_analytics():
    # Read Bronze table as streaming
    bronze_df = dlt.read_stream("bronze_hr_analytics")

    # Fill numeric NULLs with 0
    numeric_cols = get_numeric_cols(bronze_df)
    df = bronze_df.fillna(0, subset=numeric_cols)

    # Fill string NULLs with 'UNKNOWN'
    string_cols = get_string_cols(bronze_df)
    df = df.fillna("UNKNOWN", subset=string_cols)

    # Standardize Attrition to uppercase
    df = df.withColumn("Attrition", F.upper(F.col("Attrition")))

    # Derived columns
    df = (
        df.withColumn("is_attrited", F.when(F.col("Attrition") == "YES", True).otherwise(False))
          .withColumn("AnnualIncome", F.col("MonthlyIncome") * F.lit(12))
          .withColumn(
              "TenureBucket",
              F.when(F.col("YearsAtCompany") < 3, "New")
               .when(F.col("YearsAtCompany") < 7, "Experienced")
               .otherwise("Veteran")
          )
    )

    # Drop unused columns
    df = df.drop("EmployeeCount", "StandardHours")

    return df
     